# Ноутбук 3 — оценка базовой модели (zero-shot) на GQA-ru и MMBench-ru

Меряю метрики модели `deepvk/llava-gemma-2b-lora` БЕЗ дообучения — это точка отсчёта.
Потом, после LoRA-дообучения, сравню с этими числами.

- **GQA-ru**: даю картинку + вопрос, модель отвечает словом, сравниваю с эталоном (accuracy).
- **MMBench-ru**: картинка + вопрос + варианты A/B/C/D, модель выбирает букву (accuracy).

Сначала прогоняю на 50 примерах (проверить, что всё считается верно), потом на большем объёме.

## Ставлю библиотеки

Версии те же, что в ноутбуке 2 (модель 2024 года заводится только на старом окружении).
datasets и tqdm ставлю под этот же набор. **После установки перезапусти сессию.**

In [ ]:
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "huggingface_hub==0.24.6" "accelerate==0.33.0" "datasets==2.21.0" "pillow<12"

После перезапуска — проверка версий.

In [ ]:
import torch, transformers
print(torch.__version__, transformers.__version__)
print('GPU:', torch.cuda.is_available())

## Логин в HuggingFace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Гружу модель (fp16)

In [ ]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, AutoTokenizer

model_id = 'deepvk/llava-gemma-2b-lora'
model = LlavaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map='auto')
processor = AutoProcessor.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
print('модель готова')

## Функции: спросить модель, нормализовать ответ, посчитать совпадение

`ask` — задаёт модели вопрос по картинке и возвращает короткий ответ.
Дальше — нормализация и правила совпадения для GQA и парсинг буквы для MMBench.

In [ ]:
import re

def ask(img, question, max_new=10):
    messages = [{'role': 'user', 'content': '<image>\n' + question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(images=[img], text=text, return_tensors='pt').to(model.device)
    inputs['pixel_values'] = inputs['pixel_values'].to(torch.float16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def normalize(s):
    s = str(s).lower().strip()
    s = re.sub(r'[^0-9a-zа-яё ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def gqa_correct(pred, gold):
    p, g = normalize(pred), normalize(gold)
    if not g:
        return False
    if p == g:
        return True
    if g in p.split():
        return True
    if p.startswith(g + ' ') or p.endswith(' ' + g):
        return True
    return False


def build_mmbench_options(ex):
    opts = {}
    for k in ['A', 'B', 'C', 'D']:
        v = ex.get(k)
        if v is not None and str(v).strip().lower() != 'nan' and str(v).strip() != '':
            opts[k] = v
    return opts


def parse_mmbench_letter(pred, options):
    m = re.search(r'[ABCD]', str(pred).upper())
    if m and m.group(0) in options:
        return m.group(0)
    pn = normalize(pred)
    for letter, txt in options.items():
        tn = normalize(txt)
        if tn and (pn == tn or tn in pn):
            return letter
    return None

## GQA-ru: загрузка

Вопросы лежат в конфиге `testdev_balanced_instructions`, картинки — в `testdev_balanced_images`.
Связываю их по id картинки.

In [ ]:
from datasets import load_dataset

gqa_q = load_dataset('deepvk/GQA-ru', 'testdev_balanced_instructions', split='testdev')
gqa_img = load_dataset('deepvk/GQA-ru', 'testdev_balanced_images', split='testdev')
id2img = {r['id']: r['image'] for r in gqa_img}
print('вопросов:', len(gqa_q), '| картинок:', len(id2img))
print('пример:', gqa_q[0]['question'], '->', gqa_q[0]['answer'])

### Мини-прогон на 50 вопросах (проверка)

In [ ]:
from tqdm.auto import tqdm

def eval_gqa(n, show=8):
    correct = 0
    total = 0
    shown = 0
    for ex in tqdm(gqa_q.select(range(n)), total=n):
        img = id2img.get(ex['imageId'])
        if img is None:
            continue
        pred = ask(img, ex['question'] + '\nОтветь одним словом.', max_new=10)
        ok = gqa_correct(pred, ex['answer'])
        correct += int(ok)
        total += 1
        if shown < show:
            print(f"[{'+' if ok else '-'}] {ex['question']} | эталон: {ex['answer']} | ответ: {pred}")
            shown += 1
    acc = correct / total if total else 0.0
    print(f'\nGQA accuracy на {total}: {acc:.3f}')
    return acc, total

gqa_acc_mini, _ = eval_gqa(50)

### Полный прогон GQA

`N_GQA` — сколько вопросов брать. По умолчанию 1000, чтобы уложиться в сессию.
Можно увеличить (максимум ~12 216) или поставить `len(gqa_q)` для всего набора.
Важно: то же число нужно взять при оценке ПОСЛЕ обучения, чтобы сравнение было честным.

In [ ]:
N_GQA = 1000
gqa_acc, gqa_total = eval_gqa(N_GQA, show=0)

## MMBench-ru: загрузка

Один сплит `dev`, картинки встроены, ответ — буква A/B/C/D.

In [ ]:
mmb = load_dataset('deepvk/MMBench-ru', split='dev')
print('примеров:', len(mmb))
print('пример:', mmb[0]['question'], '| ответ:', mmb[0]['answer'])

### Мини-прогон на 50 вопросах (проверка)

In [ ]:
def eval_mmbench(n, show=8):
    correct = 0
    total = 0
    shown = 0
    for ex in tqdm(mmb.select(range(n)), total=n):
        opts = build_mmbench_options(ex)
        if not opts:
            continue
        q = ex['question']
        if ex.get('hint') and str(ex['hint']).strip().lower() != 'nan':
            q = str(ex['hint']) + '\n' + q
        opts_text = '\n'.join(f'{k}. {v}' for k, v in opts.items())
        prompt = q + '\n' + opts_text + '\nОтветь только буквой правильного варианта.'
        pred = ask(img=ex['image'], question=prompt, max_new=5)
        letter = parse_mmbench_letter(pred, opts)
        ok = (letter == ex['answer'])
        correct += int(ok)
        total += 1
        if shown < show:
            print(f"[{'+' if ok else '-'}] эталон: {ex['answer']} | буква: {letter} | ответ: {pred!r}")
            shown += 1
    acc = correct / total if total else 0.0
    print(f'\nMMBench accuracy на {total}: {acc:.3f}')
    return acc, total

mmb_acc_mini, _ = eval_mmbench(50)

### Полный прогон MMBench

`N_MMB` — сколько брать (максимум 3910). Одинаковое число до и после обучения.

In [ ]:
N_MMB = 1000
mmb_acc, mmb_total = eval_mmbench(N_MMB, show=0)

## Итоговые числа базовой модели

In [ ]:
import json
baseline = {
    'gqa': {'accuracy': round(gqa_acc, 4), 'n': gqa_total},
    'mmbench': {'accuracy': round(mmb_acc, 4), 'n': mmb_total},
}
print(json.dumps(baseline, ensure_ascii=False, indent=2))

with open('baseline_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(baseline, f, ensure_ascii=False, indent=2)
print('\nсохранил в baseline_metrics.json (скачай через панель Файлы слева и положи в results/)')

## Итог

Это метрики ДО обучения — база для сравнения. Числа (`baseline_metrics.json`) кладу в `results/`.
Дальше: ноутбук 4 — LoRA-дообучение на подвыборке LLaVA-Instruct-ru, затем повтор этой же оценки
с тем же N_GQA / N_MMB и таблица до/после.